In [1]:
!pip -q install duckdb datasets huggingface_hub pandas pyarrow

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

from huggingface_hub import login

login(HF_TOKEN)

In [3]:
# downloading the dataset
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id = "FlyRank/internship-warehouse",
    repo_type = "dataset"
)
print(dataset_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [4]:
# connecting DuckDB
import duckdb
con = duckdb.connect()

import os
for root, dirs, files in os.walk(dataset_path):
  for file in files:
    print(os.path.join(root,file))

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/README.md
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/.gitattributes
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_query_90d.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily

In [5]:
con.sql("""
CREATE VIEW dim_content AS
SELECT *
FROM read_parquet(
'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet'
)
""")

con.sql("""
CREATE VIEW dim_clients AS
SELECT *
FROM read_parquet(
'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet'
)
""")

con.sql("""
CREATE VIEW fact_content_query_90d AS
SELECT *
FROM read_parquet(
'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_query_90d.parquet'
)
""")

con.sql("""
CREATE VIEW fact_content_daily_performance AS
SELECT *
FROM read_parquet(
'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/*/*.parquet',
hive_partitioning=true
)
""")


### **Dataset Inspection**

In [6]:
con.sql("SHOW TABLES").df()

,name
0,dim_clients
1,dim_content
2,fact_content_daily_performance
3,fact_content_query_90d


In [7]:
con.sql("DESCRIBE fact_content_daily_performance").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [8]:
con.sql("DESCRIBE fact_content_query_90d").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [10]:
con.sql("DESCRIBE dim_clients").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [12]:
con.sql("SELECT * FROM dim_content LIMIT 10").df()

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False
5,client_04660893ae39614a,content_01fc9e2e57898b55,keyword_5f10a495acae452f,url_246e01c4def02cc4,26,4,103,2026-05-30,2026-07-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,16387,2672,NaT,NaT,True,False
6,client_04660893ae39614a,content_0212158fa61c5fcb,keyword_d09961a323fa98cc,url_ca7f2d063e633e4b,24,5,105,2026-05-29,2026-07-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,14383,2396,NaT,NaT,True,False
7,client_04660893ae39614a,content_023d807c7d922db1,keyword_009040d227d75d60,url_f59d28939e7cdbac,34,7,76,2026-06-13,2026-06-13,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,11513,1929,NaT,NaT,True,False
8,client_04660893ae39614a,content_026f7405cc253242,keyword_44ce19fbf6ff2681,url_27e808168e45d073,18,4,100,2026-06-10,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15296,2481,NaT,NaT,True,False
9,client_04660893ae39614a,content_02c8b23ea5bdb275,keyword_304b740d397b3f6e,url_efe4872b9f40eb96,28,5,104,2026-06-11,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,14576,2357,NaT,NaT,True,False


# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

##### > One row represents the daily performance metrics of one content page for one client on one reporting date.

- Each row is uniquely identified by `report_date`,`client_hash_id`,`content_hash_id`

##### > Since my lane is Refresh/Content Opportunity Scoring so the tables I would use are `fact_content_daily_performance` as the primary table and `dim_content` to enrich it with content metadata.

##### > I will use data from March 2026 because it is a mid-panel month that can be used for feature development without using the final evaluation month (June 2026).

##### > I will rank content pages by their refresh priority using current content and performance metrics as a proxy for refresh opportunity.
- I am ranking pages for refresh because the warehouse contains current performance and content attributes that can be used to estimate refresh priority.

##### > I will deliberately exclude future performance information and any label-derived fields because they would leak information that is not available when deciding whether a page should be refreshed and thus preventing the data leakage chances.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## **features**

| Field | Bucket | Why? |
|------|--------|------|
| `gsc_impressions` | Feature | Measures the search visibility of a content page and helps identify pages with low organic exposure. |
| `gsc_clicks` | Feature | Indicates how much user traffic the page receives from Google Search. |
| `gsc_avg_position` | Feature | Represents the average search ranking of the page, which is an important indicator of search performance. |
| `content_updated_date` | Feature | Indicates how recently the content was updated and helps identify potentially stale pages. |
| `search_volume` | Feature | Represents the search demand for the target keyword and helps prioritize high-impact refresh opportunities. |
| `content_type` | Feature | Provides contextual information about the type of content, which may influence refresh strategy. |
| **Refresh Priority Ranking** | Label / Proxy | The objective is to rank content pages by their refresh priority using current search performance and content metadata as proxies for refresh opportunity. |
| `report_date` | Context | Identifies the reporting date for each observation but is not used directly as a predictive feature. |
| `client_hash_id` | Context | Identifies the client associated with each content page and is used only for identification and joins. |
| `content_hash_id` | Context | Identifies the content page and is used only for joins and record identification. |
| Future performance metrics | Excluded | Not available at decision time and would introduce data leakage. |
| Label-derived features | Excluded | Would directly leak information about the prediction target and produce unrealistic model performance. |
| June 2026 evaluation data | Excluded | Reserved as an unseen evaluation period and therefore excluded from feature development. |

- ##### The selected features describe the search visibility and characteristics of each content page that would be available when deciding whether a page should be refreshed. Identifiers are retained only for joining and tracking records, while future information is excluded to prevent leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### **Query 1 - Verfiy the Grain**

In [14]:
con.sql(" SELECT report_date,client_hash_id,content_hash_id FROM fact_content_daily_performance WHERE month = '2026-03' LIMIT 10;").df()

,report_date,client_hash_id,content_hash_id
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8


The sample shows that each row corresponds to a unique combination of reporting date, client, and content page, matching the expected grain.

### **Query 2 - Verfiy counts**

In [15]:
con.sql("SELECT COUNT(*) AS total_rows, MIN(report_date) AS start_date, MAX(report_date) AS end_date FROM fact_content_daily_performance WHERE month = '2026-03';").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


This confirms the number of observations available for the March 2026 modeling window.

### **Query 3 — Verify missing values / availability**

In [18]:
con.sql("SELECT COUNT(*) AS total_rows, SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available, SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available FROM fact_content_daily_performance WHERE month='2026-03';").df()

,total_rows,gsc_available,ga4_available
0,9841378,3611061.0,413966.0


The availability flags show that not every observation contains complete GSC and GA4 data, which must be considered during feature selection.

### **Query 4 — Verify the window**

In [20]:
con.sql("SELECT MIN(report_date) AS start_date, MAX(report_date) AS end_date FROM fact_content_daily_performance WHERE month='2026-03';").df()

,start_date,end_date
0,2026-03-01,2026-03-31


The date range confirms that the selected modeling window covers March 2026.

## Five-Feature Frame

The following features are selected for the refresh-priority ranking task. Each feature is available at prediction time and does not rely on future information.

| Feature | Available at Prediction Time? | Justification |
|---------|-------------------------------|---------------|
| `gsc_impressions` |  Yes | Historical search visibility is already known when deciding whether a page should be refreshed. |
| `gsc_clicks` |  Yes | Historical organic clicks are available before making the refresh decision. |
| `gsc_avg_position` |  Yes | The average search position reflects current search performance and is known at prediction time. |
| `search_volume` |  Yes | Search volume is keyword metadata that exists independently of future page performance. |
| `content_updated_date` |  Yes | The last update date is known before prediction and helps identify stale content. |

These features were selected because they describe the current state of the content page and are all available when deciding whether a page should be refreshed. None of them depends on future performance, making them suitable for building an honest predictive model.

## Leakage Trap

To demonstrate data leakage, I intentionally considered adding a label-derived feature such as future click performance (for example, clicks observed after the prediction window).

Because this information would already contain knowledge about the outcome I am trying to predict, a model using this feature would achieve an unrealistically high evaluation score.

Although the score would appear much better, it would not represent real-world performance because future click information is unavailable when deciding whether a page should be refreshed.

Therefore, the leaking feature is removed, and only features available at prediction time are retained in the final feature set.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset has several limitations that should be considered when interpreting the results:

- **Incomplete analytics coverage:** Some observations do not contain both GSC and GA4 data, reducing the number of fully usable records.
- **Uneven historical coverage:** Content pages have different creation and update dates, so some pages have much longer histories than others.
- **Limited observation period:** The analysis uses March 2026 for modeling while reserving June 2026 for evaluation, so conclusions are limited to the available time window.
- **External factors are not captured:** The warehouse records search performance but cannot explain changes caused by Google algorithm updates, seasonality, competitor actions, or marketing campaigns.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.